# 14. Human Pose - yolov8s_pose

| 항목 | 내용 |
|---|---|
| 대상 모델 | `yolov8s_pose.dxnn` |
| 카테고리 | **Human Pose** |
| 기준 환경 | **DX-All-Suite 2.3.0 / Sync 흐름** |

## 학습 목표

1. 채널 축이 앞에 오는 출력을 후보 목록으로 돌리는 판정과 변환을 수행할 수 있다.
2. 한 후보의 56개 값을 사각형·사람 점수·관절 좌표로 나누어 설명할 수 있다.
3. 사람 점수와 관절 점수가 별개임을 실행 결과의 값으로 제시할 수 있다.
4. 관절 점수가 낮은 점이 그림에서 빠지는 이유와 그 결과를 지목할 수 있다.

## 1. Human Pose (사람 자세 추정)

**영상에서 사람을 찾고 각 사람의 관절 위치를 좌표로 답하는 작업**

검출과 관절점 검출을 한 번에 한다. 사람마다 사각형 하나와 정해진 개수의 관절 좌표가 함께 나오므로, 여러 명이
있어도 **누구의 팔인지 섞이지 않는다.**

**출력의 성격**

사람 수만큼의 목록이고 사람마다 관절 개수는 고정이다. 관절이 가려져도 좌표는 나오므로 **관절마다 붙는 점수로
걸러야** 한다. 사람이 있을 점수와 관절이 보일 점수는 서로 다른 값이다.

**활용 사례**

- 운동 자세 교정·반복 횟수 세기
- 낙상·이상 행동 감지
- 동작 인식의 앞단

**포함되지 않는 작업**

- 누구인지 판정 (Person Re-ID)
- 사람 윤곽을 픽셀 단위로 나누기 (Instance Segmentation)
- 손가락 관절 (Hand Landmark)

## 2. 전체 파이프라인

```text
sample_people.jpg  640×640 BGR
  → 전처리         BGR→RGB · 비율 유지 (이미 640×640 이라 여백 0)
  → run()          입력 Tensor 하나를 전달
  → npu_0          스케일별 중간 tensor 9개 (내부 전달, Public 아님)
  → cpu_0          output0  [1, 56, 8400]  float32  (Public Output — 채널이 앞)
  → 후처리         transpose → 점수 임계값 → 중심좌표를 모서리로 → NMS → 좌표 복원
  → Result         PoseResult (box · confidence · 관절 17개)
  → 시각화         사각형·뼈대 19개 연결·관절 17개를 원본 위에 그림
```

![14. Human Pose 전체 파이프라인](resources/14_Human_Pose_pipeline.png)

## 3. 입력과 출력

### 3.1 입력

| 속성 | 값 |
|---|---|
| Tensor 이름 | `images` |
| Shape / Dtype | `[1, 640, 640, 3]` / `uint8` |
| 채널 순서 | NHWC |
| Color / 값 범위 | RGB / 0~255 |
| Resize 방식 | **Letterbox — 비율 유지 후 여백 채우기** |
| Normalization | 없음 |

**이번 sample 은 이미 640×640 이라 여백이 생기지 않는다.** `gain` 이 `1.0` 이고 `pad_x`·`pad_y` 가 0 이므로
좌표 복원이 사실상 없는 셈이다. 다른 비율의 사진을 넣으면 여백이 생기고, 그때는 여백을 빼고 `gain` 으로 나누는
계산이 그대로 필요하다.

### 3.2 출력

이 모델은 Task 가 둘이며 **여러 갈래의 중간 값을 하나의 후보 목록으로 합치는 일이 CPU Task 의 몫**이다.

| 생성 Task | Tensor | shape / dtype | 다음 사용자 | Public |
|---|---|---|---|---|
| `npu_0` | `cv2.*` 3개 | `[1, 64, 격자]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `npu_0` | `cv3.*` 3개 | `[1, 1, 격자]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `npu_0` | `cv4.*` 3개 | `[1, 51, 격자]` / `float32` | `cpu_0` | 아님 — 내부 전달 |
| `cpu_0` | `output0` | `[1, 56, 8400]` / `float32` | Host 후처리 | CPU Task 출력 (= Public 출력) |

중간 tensor 가 세 갈래(`cv2`·`cv3`·`cv4`)로 나뉘어 있고 각각 스케일 셋을 갖는다. 사각형용 64채널, 사람 점수용
1채널, 관절용 51채널이 따로 계산된 뒤 `cpu_0` 에서 하나로 합쳐진다.

**후보 8,400개는 세 격자의 합이다.** `80×80 + 40×40 + 20×20 = 6400 + 1600 + 400 = 8,400` 이며, YOLOv8 은 앵커를
쓰지 않으므로 자리마다 후보가 하나다. 12번(YOLOv5)이 자리당 3개씩 25,200개였던 것과 대비된다.

**shape 각 자리의 뜻**

| index | 값 | 뜻 |
|---|---:|---|
| 0 | 1 | 한 번에 처리하는 사진 장수 |
| 1 | 56 | 후보 하나의 항목 수 |
| 2 | 8400 | 후보 수 |

**채널이 후보보다 앞에 온다.** 앞 모델들은 `[후보, 항목]` 순서였는데 이 출력은 `[항목, 후보]` 다. 후처리는 두 축
중 **작은 쪽을 항목 축으로 보고** 필요하면 뒤집는다. 8400 과 56 을 비교해 56 쪽이 항목이라고 판정하는 식이며,
이렇게 하면 후보 수가 다른 배포본에서도 같은 코드가 동작한다.

**한 후보 56개 값의 자리**

| index | 개수 | 내용 |
|---:|---:|---|
| 0~3 | 4 | 중심 좌표와 크기 `cx, cy, w, h` (입력 640 기준) |
| 4 | 1 | 사람이 있을 점수 |
| 5~55 | 51 | 관절 17개 × `(x, y, 점수)` |

**부류 점수가 없다.** 사람 한 부류만 다루므로 12번의 80개 부류 점수 자리가 없고, 점수 하나가 곧 최종 점수다.
`conf = obj × cls` 같은 곱셈도 없다.

**관절 51개 값은 세 개씩 끊어 읽는다.**

```text
[ x0 y0 s0  x1 y1 s1  …  x16 y16 s16 ]
  └ 관절 0 ┘ └ 관절 1 ┘   └ 관절 16 ┘
```

**사람 점수와 관절 점수는 별개다.** 이번 실행의 1위는 사람 점수 `0.9469` 인데 그 사람의 코는 `0.3782`,
왼쪽 손목은 `0.3541` 이다. 사람은 확실히 있지만 그 관절이 가려졌거나 흐릿하다는 뜻이며, **사람 점수가 높다고
모든 관절 좌표를 믿을 수 없다.**

**17개 관절의 번호와 부위**

번호의 정본은 실행 코드가 들고 있는 이름 배열이며 COCO 사람 관절 정의를 따른다.

| index | 부위 | index | 부위 | index | 부위 |
|---:|---|---:|---|---:|---|
| 0 | nose | 6 | right_shoulder | 12 | right_hip |
| 1 | left_eye | 7 | left_elbow | 13 | left_knee |
| 2 | right_eye | 8 | right_elbow | 14 | right_knee |
| 3 | left_ear | 9 | left_wrist | 15 | left_ankle |
| 4 | right_ear | 10 | right_wrist | 16 | right_ankle |
| 5 | left_shoulder | 11 | left_hip | | |

세로로 읽는 표가 아니다. 0~16 을 여섯 개씩 세 묶음으로 옆에 늘어놓았다.

전체 목록: [ultralytics coco-pose.yaml](https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/coco-pose.yaml)

**`left`·`right` 의 기준은 이 sample 로 확인하지 못했다.** 인물이 정면을 향하고 있지 않아 화면 기준인지 인물
기준인지 좌표로 가려낼 수 없다. 다른 모델에서 성립한 기준을 옮겨 오지 않고 **확인되지 않았다고 적는다.**

#### `.dxnn` 파일에서 직접 확인

`parse_model` 은 `.dxnn` 안의 tensor 이름·shape·dtype·크기·Task 구성을 그대로 보고한다. `npu_0` 이 세 갈래
중간 tensor 를 내고 `cpu_0` 이 `[1, 56, 8400]` 하나로 합치는지 아래 출력에서 확인한다.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "assets" / "models").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
MODEL_PATH = REPO_ROOT / "assets" / "models" / "yolov8s_pose.dxnn"
assert MODEL_PATH.is_file(), f"모델이 없다: {MODEL_PATH}"

os.environ.setdefault("DX_ALL_SUITE", os.path.expanduser("~/Workspace/dx-all-suite"))
DX_ALL_SUITE = Path(os.environ["DX_ALL_SUITE"])

DX_APP = DX_ALL_SUITE / "dx-runtime" / "dx_app"
assert DX_APP.is_dir(), f"dx-all-suite 가 없다: {DX_APP} — 환경변수 DX_ALL_SUITE 로 위치를 알려 준다."

parsed = subprocess.run(["parse_model", "-m", str(MODEL_PATH), "-v"], capture_output=True, text=True)
print((parsed.stdout + parsed.stderr).strip())

## 4. 실습

### 4.1 실행 환경 구성

라이브러리·그림 출력 방식·입력 파일 경로를 정하고 관절 이름·뼈대 연결·색을 정의한다. 이후 셀은 여기서 만든
이름을 그대로 쓴다.

In [ ]:
import sys
import time
from dataclasses import dataclass, field

import matplotlib

# 노트북에서는 결과 그림이 셀 아래 남도록 inline, 스크립트로 단독 실행할 때는 화면이 없으므로
# 파일 기반 Agg 를 쓴다. 지정하지 않으면 GUI 백엔드를 찾다가 실패한다.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    matplotlib.use("Agg")

# 렌더링 백엔드를 dx_engine 보다 먼저 적재한다. matplotlib 은 첫 그림을 그릴 때에야 백엔드를
# 불러오는데, 그 시점이 dx_engine 적재 뒤이면 ft2font 확장 초기화가 실패한다
# (RuntimeError: module 'matplotlib.ft2font' has no attribute '__path__').
import matplotlib.backends.backend_agg  # noqa: E402,F401
import matplotlib.pyplot as plt  # noqa: E402

import cv2  # noqa: E402
import numpy as np  # noqa: E402
from dx_engine import InferenceEngine  # noqa: E402

sys.dont_write_bytecode = True
IMAGE_PATH = DX_APP / "sample" / "img/sample_people.jpg"

KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]
SKELETON = [
    [15, 13], [13, 11], [16, 14], [14, 12], [11, 12], [5, 11], [6, 12], [5, 6],
    [5, 7], [6, 8], [7, 9], [8, 10], [1, 2], [0, 1], [0, 2], [1, 3], [2, 4],
    [3, 5], [4, 6],
]
LIMB_COLORS = ([(51, 153, 255)] * 4 + [(255, 51, 255)] * 3 + [(255, 128, 0)] * 5
               + [(0, 255, 0)] * 7)
KEYPOINT_COLORS = ([(0, 255, 0)] * 5 + [(255, 128, 0)] * 6 + [(51, 153, 255)] * 6)

print("keypoints", len(KEYPOINT_NAMES), "/ skeleton", len(SKELETON), "연결")

#### 결과 객체와 실행 지원 함수

- `PreprocessContext` — 전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. `scale`(gain)·`pad_x`·`pad_y` 가
  좌표 복원의 근거다
- `Keypoint` · `PoseResult` — 후처리가 만들어 시각화로 넘기는 결과. 사각형·사람 점수·관절 17개를 담는다
- `notebook_describe_array` — 배열의 shape·dtype·값 범위 출력
- `notebook_show_bgr` — 이미지 inline 표시. 3채널은 RGB 로 되돌림(안 하면 색이 뒤집힘), grayscale 은
  값 범위를 0~255 로 고정(안 하면 대비가 실제보다 강해짐)

In [ ]:
@dataclass
class PreprocessContext:
    """전처리가 원본을 어떻게 바꾸었는지 후처리에 넘기는 값. 원본 대응: common/base/i_processor.py"""

    original_width: int = 0
    original_height: int = 0
    input_width: int = 0
    input_height: int = 0
    scale: float = 1.0
    pad_x: int = 0
    pad_y: int = 0


@dataclass
class Keypoint:
    """관절 하나의 좌표와 점수. 원본 대응: common/base/i_processor.py"""

    x: float = 0.0
    y: float = 0.0
    confidence: float = 0.0


@dataclass
class PoseResult:
    """사람 하나의 사각형·점수·관절 목록. 원본 대응: common/base/i_processor.py"""

    box: list = field(default_factory=list)
    confidence: float = 0.0
    class_id: int = 0
    keypoints: list = field(default_factory=list)


def notebook_describe_array(name: str, array: np.ndarray) -> None:
    """배열의 shape·dtype·값 범위를 한 줄로 출력한다. 노트북 전용."""
    array = np.asarray(array)
    print(f"{name:16s} {str(array.shape):18s} {array.dtype!s:8s} "
          f"min={array.min():9.4f} max={array.max():9.4f} mean={array.mean():9.4f}")


def notebook_show_bgr(image: np.ndarray, title: str = "Image", figsize: tuple = (9, 6)) -> None:
    """BGR 또는 grayscale 이미지를 inline 으로 표시한다. 노트북 전용."""
    plt.figure(figsize=figsize)
    if image.ndim == 2:
        plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

#### 런타임이 보고하는 입출력 정보

`.dxnn` 을 NPU 에 올리고 런타임이 보고하는 입출력 정보를 읽는다. 출력이 `[1, 56, 8400]` 하나인 것을 확인한다.

In [ ]:
engine = InferenceEngine(str(MODEL_PATH))
input_info = engine.get_input_tensors_info()
output_info = engine.get_output_tensors_info()

print("input :", input_info)
print("output:", output_info)

INPUT_HEIGHT, INPUT_WIDTH = input_info[0]["shape"][1], input_info[0]["shape"][2]
print(f"model input (WxH) = {INPUT_WIDTH}x{INPUT_HEIGHT}")

### 4.2 전처리

비율을 유지해 줄이고 남는 자리를 회색으로 채운다. 이번 sample 은 이미 640×640 이라 줄이지도 채우지도 않지만,
코드는 다른 비율의 사진에도 같게 동작한다.

확인할 것:

- 색 순서 — BGR → RGB
- 크기 — 640×640
- `gain` — 1.0 (줄이지 않음)
- 여백 — 0 (채우지 않음)

#### 입력 이미지 확인

원본 사진을 읽어 크기와 dtype 을 확인한다. 사람이 여럿 있는 장면이라 결과 개수를 눈으로 셀 수 있다.

In [ ]:
image = cv2.imread(str(IMAGE_PATH))

print(f"shape {image.shape} dtype {image.dtype}")
notebook_show_bgr(image, f"Input - {IMAGE_PATH.name}", figsize=(8, 8))

#### 전처리 함수 정의

- `pose_preprocessor_create` — 모델 입력 크기와 여백 색을 state 에 담음
- `pose_letterbox` — 비율 유지 축소와 여백 채우기
- `pose_preprocessor_process` — letterbox 를 적용하고 복원용 값을 Context 에 담음

In [ ]:
def pose_preprocessor_create(input_width: int, input_height: int,
                             pad_color: tuple = (114, 114, 114)) -> dict:
    """모델 입력 크기와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_preprocessor()"""
    return {"input_width": input_width, "input_height": input_height, "pad_color": pad_color}


def pose_letterbox(state: dict, image: np.ndarray) -> tuple:
    """비율을 유지해 줄이고 남는 자리를 채운다. 원본 대응: LetterboxPreprocessor._letterbox()"""
    height, width = image.shape[:2]
    gain = min(state["input_height"] / height, state["input_width"] / width)
    new_size = int(round(width * gain)), int(round(height * gain))

    delta_w = (state["input_width"] - new_size[0]) / 2
    delta_h = (state["input_height"] - new_size[1]) / 2
    if (width, height) != new_size:
        image = cv2.resize(image, new_size, interpolation=cv2.INTER_LINEAR)

    top, bottom = int(round(delta_h - 0.1)), int(round(delta_h + 0.1))
    left, right = int(round(delta_w - 0.1)), int(round(delta_w + 0.1))
    padded = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT,
                                value=state["pad_color"])
    return padded, gain, (top, left)


def pose_preprocessor_process(state: dict, input_image: np.ndarray) -> tuple:
    """이미지를 입력 Tensor 와 Context 로 바꾼다. 원본 대응: Runner.preprocess()"""
    ctx = PreprocessContext()
    ctx.original_height, ctx.original_width = input_image.shape[:2]
    ctx.input_width = state["input_width"]
    ctx.input_height = state["input_height"]

    rgb = cv2.cvtColor(input_image, cv2.COLOR_BGR2RGB)
    padded, gain, pad = pose_letterbox(state, rgb)
    ctx.scale = gain
    ctx.pad_y, ctx.pad_x = pad
    return padded, ctx

#### 전처리 실행

입력 Tensor 와 Context 를 만든다. `gain` 이 1.0 이고 여백이 0 인 것을 확인한다.

In [ ]:
preprocessor_state = pose_preprocessor_create(INPUT_WIDTH, INPUT_HEIGHT)
input_tensor, ctx = pose_preprocessor_process(preprocessor_state, image)

print(f"input_tensor  shape={input_tensor.shape} dtype={input_tensor.dtype} "
      f"contiguous={input_tensor.flags['C_CONTIGUOUS']}")
print(f"context       original={ctx.original_width}x{ctx.original_height} "
      f"gain={ctx.scale:.6f} pad_x={ctx.pad_x} pad_y={ctx.pad_y}")

### 4.3 추론

#### DXNN Inference 실행

입력 Tensor 를 넘기고 Public Output 을 받는다. 출력되는 시간은 1회 측정값이고 런타임과의 왕복 비용이 포함돼
있어 초당 처리 장수로 환산하지 않는다.

In [ ]:
started = time.perf_counter()
outputs = engine.run([input_tensor])
inference_ms = (time.perf_counter() - started) * 1000.0

print(f"inference: {inference_ms:.2f} ms")
print("output count =", len(outputs))
for index, output in enumerate(outputs):
    array = np.asarray(output)
    print(index, array.shape, array.dtype, f"min={array.min():.4f}", f"max={array.max():.4f}")

#### Raw Output 직접 검사

값에서 직접 확인할 것:

- 두 축의 크기 — 어느 쪽이 항목이고 어느 쪽이 후보인지
- 뒤집은 뒤의 shape — `[8400, 56]`
- 사람 점수 범위와 임계값 통과 개수
- 항목 구성 — `4 + 1 + 51 = 56`

In [ ]:
raw = np.squeeze(np.asarray(outputs[0]))
notebook_describe_array("output0", raw)

print(f"두 축 {raw.shape} → 작은 쪽({min(raw.shape)})을 항목 축으로 본다")
candidates = np.transpose(raw) if raw.shape[0] < raw.shape[1] else raw
print(f"후보 목록 shape = {candidates.shape}  (후보 {candidates.shape[0]:,}개 × 항목 "
      f"{candidates.shape[1]}개)")
print(f"항목 구성 = 4(box) + 1(person score) + {len(KEYPOINT_NAMES)}×3(keypoint) = "
      f"{4 + 1 + len(KEYPOINT_NAMES) * 3}")

scores = candidates[:, 4]
SCORE_THRESHOLD = 0.3
print(f"\n사람 점수 {scores.min():.6f} ~ {scores.max():.6f}")
print(f"{SCORE_THRESHOLD} 초과 후보 {int((scores >= SCORE_THRESHOLD).sum())}개 / "
      f"{candidates.shape[0]:,}개")
print(f"후보 8400 = 80×80 + 40×40 + 20×20 = {80 * 80 + 40 * 40 + 20 * 20:,} (자리당 후보 1개)")

### 4.4 후처리

축을 뒤집어 후보 목록으로 만들고, 사람 점수로 걸러 내고, 중심 좌표를 모서리로 바꾸어 겹침을 정리한 뒤, 사각형과
관절 좌표를 모두 원본 좌표로 되돌린다.

**관절 좌표도 같은 방법으로 되돌린다.** 사각형만 되돌리고 관절을 그대로 두면 관절이 사각형 밖으로 벗어난다.
`(값 - 여백) / gain` 을 좌표마다 적용하며, 관절 점수는 좌표가 아니므로 손대지 않는다.

**관절 시작 자리를 뒤에서부터 센다.** 항목 수에서 관절 값 개수(`17 × 3 = 51`)를 빼 시작 자리를 구하므로, 부류
번호가 하나 더 붙은 다른 배포본에서도 같은 코드가 맞는 자리를 찾는다.

#### 후처리 함수 정의

- `pose_postprocessor_create` — 임계값과 관절 개수를 state 에 담음
- `pose_postprocessor_process` — 축 뒤집기·걸러내기·NMS·좌표 복원을 거쳐 Result 생성

In [ ]:
def pose_postprocessor_create(config: dict) -> dict:
    """Config 와 고정 상수를 state 에 담는다. 원본 대응: Factory.create_postprocessor()"""
    return {
        "score_threshold": config.get("score_threshold", 0.3),
        "nms_threshold": config.get("nms_threshold", 0.45),
        "num_keypoints": config.get("num_keypoints", 17),
    }


def pose_postprocessor_process(state: dict, outputs: list, ctx: PreprocessContext) -> list:
    """Raw Output 을 Result 로 변환한다. 원본 대응: Runner.postprocess()"""
    squeezed = np.squeeze(np.asarray(outputs[0]))
    if squeezed.ndim < 2:
        return []
    # 항목 축이 앞에 오면 뒤집어 [후보, 항목] 으로 맞춘다.
    output = np.transpose(squeezed) if squeezed.shape[0] < squeezed.shape[1] else squeezed

    keypoint_values = state["num_keypoints"] * 3
    keypoint_start = output.shape[1] - keypoint_values

    scores = output[:, 4]
    kept = scores >= state["score_threshold"]
    if not np.any(kept):
        return []

    filtered = output[kept]
    filtered_scores = scores[kept]
    keypoints_raw = filtered[:, keypoint_start:]

    corners = np.column_stack([
        filtered[:, 0] - filtered[:, 2] * 0.5, filtered[:, 1] - filtered[:, 3] * 0.5,
        filtered[:, 0] + filtered[:, 2] * 0.5, filtered[:, 1] + filtered[:, 3] * 0.5,
    ])
    boxes_xywh = np.column_stack([corners[:, 0], corners[:, 1],
                                  corners[:, 2] - corners[:, 0], corners[:, 3] - corners[:, 1]])
    indices = cv2.dnn.NMSBoxes(boxes_xywh.tolist(), filtered_scores.tolist(),
                               state["score_threshold"], state["nms_threshold"])
    if len(indices) == 0:
        return []

    gain = max(ctx.scale, 1e-6)
    results = []
    for index in np.array(indices).reshape(-1):
        box = corners[index]
        points = [
            Keypoint(x=float(np.clip((point[0] - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
                     y=float(np.clip((point[1] - ctx.pad_y) / gain, 0, ctx.original_height - 1)),
                     confidence=float(point[2]))
            for point in keypoints_raw[index].reshape(-1, 3)
        ]
        results.append(PoseResult(
            box=[float(np.clip((box[0] - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
                 float(np.clip((box[1] - ctx.pad_y) / gain, 0, ctx.original_height - 1)),
                 float(np.clip((box[2] - ctx.pad_x) / gain, 0, ctx.original_width - 1)),
                 float(np.clip((box[3] - ctx.pad_y) / gain, 0, ctx.original_height - 1))],
            confidence=float(filtered_scores[index]), class_id=0, keypoints=points))
    return results

#### 후처리 실행

확인할 것:

- 단계별 개수 — 8,400 → 점수 통과 → 최종
- 사람마다의 관절 개수 — 17개인지
- 사람 점수와 관절 점수의 차이 — 같은 사람 안에서 관절마다 점수가 다른지
- 점수가 낮은 관절 — 그림에서 빠질 관절이 몇 개인지

In [ ]:
postprocessor_state = pose_postprocessor_create({})
results = pose_postprocessor_process(postprocessor_state, outputs, ctx)

score_pass = int((scores >= postprocessor_state["score_threshold"]).sum())
print(f"후보 {candidates.shape[0]:,}개 → 점수 {postprocessor_state['score_threshold']} 통과 "
      f"{score_pass}개 → NMS {postprocessor_state['nms_threshold']} 통과 {len(results)}명")

for order, pose in enumerate(sorted(results, key=lambda r: -r.confidence), start=1):
    x1, y1, x2, y2 = pose.box
    low = sum(1 for point in pose.keypoints if point.confidence < 0.3)
    print(f"  #{order} person score={pose.confidence:.4f} "
          f"box=({x1:6.1f}, {y1:6.1f}, {x2:6.1f}, {y2:6.1f}) "
          f"관절 {len(pose.keypoints)}개 (0.3 미만 {low}개)")

best = max(results, key=lambda r: r.confidence)
print(f"\n1위 사람(점수 {best.confidence:.4f})의 관절:")
for index in (0, 5, 6, 9, 10, 15, 16):
    point = best.keypoints[index]
    print(f"  {index:2d} {KEYPOINT_NAMES[index]:15s} ({point.x:6.1f}, {point.y:6.1f}) "
          f"conf={point.confidence:.4f}")

### 4.5 시각화

#### Visualizer 코드

결과 객체만 받아 사각형·뼈대·관절을 그린다. **관절 점수가 0.3 에 못 미치면 그리지 않고**, 그 관절에 이어지는
뼈대도 함께 빠진다. 부위별로 색을 달리해 머리·팔·다리를 구분한다. 그림에 넣는 글자는 영문만 쓴다 — OpenCV 기본
폰트에 한글 자모가 없어 한글을 넣으면 글자가 깨진다.

In [ ]:
def pose_visualizer_visualize(image: np.ndarray, results: list, keypoint_radius: int = 5,
                              skeleton_thickness: int = 2,
                              keypoint_threshold: float = 0.3) -> np.ndarray:
    """Result 를 원본 이미지 위에 그린다. 원본 대응: Runner.visualize()"""
    output = image.copy()

    for pose in results:
        x1, y1, x2, y2 = [int(value) for value in pose.box]
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"person: {pose.confidence:.2f}"
        (label_w, label_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        label_y = y1 - 10 if y1 - 10 > label_h else y1 + 10
        cv2.rectangle(output, (x1, label_y - label_h), (x1 + label_w, label_y + label_h),
                      (0, 255, 0), cv2.FILLED)
        cv2.putText(output, label, (x1, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1,
                    cv2.LINE_AA)

        for order, (start, end) in enumerate(SKELETON):
            if (pose.keypoints[start].confidence < keypoint_threshold
                    or pose.keypoints[end].confidence < keypoint_threshold):
                continue
            cv2.line(output,
                     (int(pose.keypoints[start].x), int(pose.keypoints[start].y)),
                     (int(pose.keypoints[end].x), int(pose.keypoints[end].y)),
                     LIMB_COLORS[order], skeleton_thickness, cv2.LINE_AA)

        for index, point in enumerate(pose.keypoints):
            if point.confidence < keypoint_threshold:
                continue
            cv2.circle(output, (int(point.x), int(point.y)), keypoint_radius,
                       KEYPOINT_COLORS[index], -1)

    return output

#### 결과 시각화

자세 결과를 inline 으로 표시한다. 뼈대가 사람 형태로 이어지는지, 점수가 낮은 관절이 빠져 뼈대가 끊긴 자리가
있는지 확인한다.

In [ ]:
output_image = pose_visualizer_visualize(image, results)
notebook_show_bgr(output_image, f"{len(results)} people with 17 keypoints", figsize=(9, 9))

#### 관절 점수 임계값을 바꿔 보기

관절 점수 임계값을 올리면 흐릿한 관절이 빠지고 뼈대가 끊긴다. 세 값으로 그려 나란히 놓으면 임계값이 그림을
어떻게 바꾸는지 보인다.

In [ ]:
panels = []
for threshold in (0.0, 0.3, 0.8):
    drawn = pose_visualizer_visualize(image, results, keypoint_threshold=threshold)
    count = sum(1 for pose in results for point in pose.keypoints
                if point.confidence >= threshold)
    cv2.putText(drawn, f"kpt thr {threshold:.1f} - {count} points", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    panels.append(cv2.resize(drawn, (420, int(420 * image.shape[0] / image.shape[1]))))

print("임계값을 올릴수록 그려지는 관절 수가 줄고 뼈대가 끊긴다.")
notebook_show_bgr(cv2.hconcat(panels), "Keypoint threshold 0.0 / 0.3 / 0.8", figsize=(14, 5))